## Section 4 - Task 4 (Additional Feature) - Sentiment Analysis

**Environment:** Python 3, Jupyter Notebook

**Libraries used:**
- `gradio`, `plotly`: interface and visualizations
- `pandas`: data processing
- `collections`, `typing`, `math`: supporting libraries
---

## Introduction
This notebook explains the functionalities of task 4 feature - sentiment analysis - using the code from these files - `data_loader.py`, `sentiment.py`, `sentiment_tab.py`.

Sentiment analysis in this website targets brands, showing and comparing customers' opinions on one or multiple cosmetic products. Brands can use the analysis outcome from this feature to support their product strategies.

The sentiment analysis uses approximately **60,000 review text + metadata** information **embedded (unweighted)** using Glove 6B 300 dimensions and trained on a **Random Forest Classifier** machine learning model. This is also the best-performing configuration in Milestone 1 (F1 = 0.6808 across 36 tested combinations).

In [ ]:
import math
import plotly.graph_objects as go
import pandas as pd
from collections import Counter
from typing import Dict, List, Tuple, Optional

# Additional libraries in ui/sentiment_tab.py
import gradio as gr
from plotly.subplots import make_subplots
from ui.components import render_key_terms_table
from core.sentiment import (
    get_sentiment_breakdown,
    get_key_terms,
    create_multi_comparison_chart,
)
from core.data_loader import new_reviews

### 2 - Approach

Originally, two approaches are considered for the backbone of the sentiment analysis:
- **Approach A - Rating-based**: This approach uses review rating threshold to classify customers' opinions on a product (>=4 for positive, 3 for neutral, and <=2 for negative)
- **Approach B - Text-based**: This approach uses the predict probability of buyers from the best-performing model in milestone 1 (RF + unweighted Glove + metadata)

**Approach B** is chosen due to:
- Milestone 1 best-performing pipeline reuse: the F1 score of `0.6808` is the strongest empirical result available
- More information: GloVe embedding + metadata provides more information than just numeric rating

**Approach A is also limited** by the rating distribution observed in Milestone 1: 78.7% of reviews are from buyers who give high ratings, so a rating-threshold sentiment classifier would show most products as 'positive', flattening differences between products. Approach B's text-based classification surfaces these differences more meaningfully.

**Predict Probability vs Predict only**
Although model C (RF + unweighted GloVe + metadata) is trained as binary classifier (buyer/non-buyer), this feature uses `predict_proba` to get probability a customer might purchase a product. This choice is made to add a new zone 'neutral' to the classifier, representing the model's uncertainty.

Probability mapping:
- **P(buyer) ≥ 0.60** → positive
- **0.40 < P(buyer) < 0.60** → neutral (model uncertainty zone)
- **P(buyer) ≤ 0.40** → negative

### 3 - Implementation
#### Overview

Flow: 
df -> RF + unweighted GloVe + metadata model -> probability score of 'buyer' prediction result -> new text_sentiment column -> cache (if first time or with new reviews added) -> sentiment.py functions -> render charts on UI

Files map:
- `data_loader.py`: precomputes text_sentiment column from Model C at app startup, caches to disk for subsequent loads
- `sentiment.py`: provides three core functions: `get_sentiment_breakdown()`, `get_key_terms()`, `create_multi_comparison_chart()`, all of which read from the precomputed column
- `sentiment_tab.py`: assembles the Gradio UI tab and wires the "Analyze" button to the backend functions

#### Pre-Computation Text Sentiment: Function `compute_text_sentiment_cached()` (`data_loader.py`)

**Caching strategy**: First-run prediction on ~60,000 reviews. The function saves the resulting sentiment array to `models/text_sentiment_cache.pkl` and loads it on subsequent app starts. The cache is invalidated automatically if its length doesn't match the current dataframe length, which is a simple safeguard against running on stale data after the source CSV changes.

**Batch prediction**: We load the entire dataframe (about 60,000 reviews) then call `predict_proba` only once, instead of looping and calling the predict function over all the reviews one at a time.

**`np.where`** is a vectorized if-else: it checks each probability and assigns a label based on the condition. Since `text_sentiment` is a single column, we nest two `np.where` calls into one variable:

- First check: Is it ≥ 0.6 (pos_threshold)? If yes → 'positive'
- Otherwise: Is it ≤ 0.4 (neg_threshold)? If yes → 'negative'. Otherwise → 'neutral'.

In [ ]:
def compute_text_sentiment_cached(
        df: pd.DataFrame,
        model_c: Any,
        label_encoder: Any,
        base_path: str,
        pos_threshold: float = 0.60,
        neg_threshold: float = 0.40,
) -> pd.DataFrame:
    cached_path = os.path.join(base_path, 'models', 'text_sentiment_cache.pkl')
    """Pre-compute text sentiments and cache."""

    # Fast path: Load from cached text sentiments if it exists and match current df size
    if os.path.exists(cached_path):
        print(f"Loading text sentiments from cache ({cached_path})...")
        start = time.time()
        with open(cached_path, 'rb') as f:
            cached = pickle.load(f)
        

        if len(cached) == len(df):
            df['text_sentiment'] = cached
            elapsed = time.time() - start
            print(f"Loaded {len(cached):,} text sentiments in {elapsed:.2f}s (cached)")
            return df
        else:
            print(f"Cache size mismatch ({len(cached)} vs {len(df)})")


    
    # Predict on entire dataframe
    print("Computing text-based sentiments for all reviews using RF + unweighted + metadata")
    print("  (First run — will cache for faster future loads)")

    start = time.time()
    input_df = pd.DataFrame({
        'processed_review_text': df['processed_review_text'].fillna('').astype(str),
        'processed_title': df['processed_title'].fillna('').astype(str) if 'processed_title' in df.columns else '',
        'price': df['price'],
        'avg_product_rating': df['avg_product_rating'],
        'product_rating_count': df['product_rating_count'],
        'review_rating': df['review_rating'],
        'brand_encoded': _encode_brands(df['brand_name'], label_encoder),
    })

    probs_buyer = model_c.predict_proba(input_df)[:,1]

    sentiments = np.where(
        probs_buyer >= pos_threshold, 'positive',
        np.where(probs_buyer <= neg_threshold, 'negative', 'neutral')
    )

    df['text_sentiment'] = sentiments

    elapsed = time.time() - start
    print(f"Computed text sentiments in {elapsed:.2f}s (cached)")

    pos = (sentiments == 'positive').sum()
    neu = (sentiments == 'neutral').sum()
    neg = (sentiments == 'negative').sum()

    total = len(sentiments)
    print(f"positive={pos:,} ({pos/total*100:.1f}%)")
    print(f"neutral={neu:,} ({neu/total*100:.1f}%)")
    print(f"negative={neg:,} ({neg/total*100:.1f}%)")

    print(f"Saving cache to {cached_path}")
    with open(cached_path, 'wb') as f:
        pickle.dump(sentiments, f, protocol=4)
    
    print("Text sentiments cache saved")

    return df

### Function `_encode_brands` (`data_loader.py`)
Model c expects an encoded `brand_name` column. This helper handles the encoding, defaulting unseen brands to 0.

Mapping unseen brands to 0 follows the same convention as `core/classifier.py`. The alternative (raising an error) would crash the precomputation if any review references a brand not seen during model training.

In [ ]:
def _encode_brands(brand_series: pd.Series, label_encoder: Any) -> np.ndarray:
    """
    Encode brand names into integers, unseen brands to 0
    """
    known_brands = set(label_encoder.classes_)
    encoded = np.zeros(len(brand_series), dtype=int)
    for i, brand in enumerate(brand_series.fillna('unknown')):
        if brand in known_brands:
            encoded[i] = label_encoder.transform([brand])[0]
        # else: leave unseen brands as 0

    return encoded

### Sentiment Breakdown: Function `get_sentiment_breakdown()` (`sentiment.py`)

This function takes a product (identified through its ID), the dataframe, and optionally a dataframe of new reviews, and returns a dictionary with counts and percentages for 3 classes (positive, neutral, negative).

`df['text_sentiment']` is used to filter and count the number of positive, neutral, and negative reviews.

Edge cases like no reviews in the dataframe and new reviews are handled by returning 0 count and percentages across all 3 classes.

New reviews are temporarily classified as 'neutral' during the current session. Properly classifying them would require running Model C (RF + unweighted GloVe + metadata) at the moment of review submission, which lives in `classifier.py`.

Submitted reviews receive proper text-based sentiment on the next app startup, when the cache regenerates.

In [ ]:
def get_sentiment_breakdown(
    product_id: str,
    df: pd.DataFrame,
    new_reviews: Optional[pd.DataFrame] = None
) -> Dict:
    """
    Calculate sentiment distribution for a product's reviews.

    Args:
        product_id: Product identifier to filter reviews
        df: Full reviews DataFrame with 'product_id' and 'text_sentiment' columns
        new_reviews: Optional DataFrame of new reviews to include

    Returns:
        Dict containing counts and percentages for positive/neutral/negative sentiment
    """
    # Filter reviews for the product
    product_reviews = df[df['product_id'] == product_id].copy()

    # Add new reviews if provided
    if new_reviews is not None and not new_reviews.empty:
        new_product_reviews = new_reviews[new_reviews['product_id'] == product_id].copy()
        product_reviews = pd.concat([product_reviews, new_product_reviews], ignore_index=True)

    if 'text_sentiment' not in product_reviews.columns:
        product_reviews['text_sentiment'] = 'neutral'
    else:
        product_reviews['text_sentiment'] = product_reviews['text_sentiment'].fillna('neutral')

    total = len(product_reviews)

    # Handle edge case: no reviews
    if total == 0:
        return {
            'positive_count': 0,
            'neutral_count': 0,
            'negative_count': 0,
            'total': 0,
            'positive_pct': 0.0,
            'neutral_pct': 0.0,
            'negative_pct': 0.0
        }

    # Classify by rating
    positive_count = len(product_reviews[product_reviews['text_sentiment'] == 'positive'])
    neutral_count = len(product_reviews[product_reviews['text_sentiment'] == 'neutral'])
    negative_count = len(product_reviews[product_reviews['text_sentiment'] == 'negative'])

    return {
        'positive_count': positive_count,
        'neutral_count': neutral_count,
        'negative_count': negative_count,
        'total': total,
        'positive_pct': (positive_count / total) * 100,
        'neutral_pct': (neutral_count / total) * 100,
        'negative_pct': (negative_count / total) * 100
    }

### Key Terms Extraction: Function `get_key_terms()` (`sentiment.py`)

This function also filters reviews by its sentiment class in the `text_sentiment` column, then count that product's review class frequency, outputting words that belong to positive or negative reviews.

`processed_review_text` is the column produced by the Milestone 1 pipeline: lowercased, tokenized with the regex pattern `[a-zA-Z]+(?:[-'][a-zA-Z]+)?`, stopwords removed, lemmatized, and joined back as space-separated tokens. Splitting on spaces is therefore enough to recover individual tokens for counting.

Top 15 is a balanced and readable number of words.

In [ ]:
def get_key_terms(
    product_id: str,
    df: pd.DataFrame,
    sentiment: str = 'positive',
    top_n: int = 15,
    new_reviews: Optional[pd.DataFrame] = None
) -> List[Tuple[str, int]]:
    """
    Extract most frequent words from reviews of a specific sentiment category.

    Args:
        product_id: Product identifier to filter reviews
        df: Full reviews DataFrame
        sentiment: 'positive' (rating >= 4), 'neutral' (rating == 3), or 'negative' (rating <= 2)
        top_n: Number of top terms to return
        new_reviews: Optional DataFrame of new reviews to include

    Returns:
        List of (word, count) tuples sorted by frequency descending
    """
    # Filter reviews for the product
    product_reviews = df[df['product_id'] == product_id].copy()

    # Add new reviews if provided
    if new_reviews is not None and not new_reviews.empty:
        new_product_reviews = new_reviews[new_reviews['product_id'] == product_id].copy()
        product_reviews = pd.concat([product_reviews, new_product_reviews], ignore_index=True)

    if 'text_sentiment' not in product_reviews.columns:
        product_reviews['text_sentiment'] = 'neutral'
    else:
        product_reviews['text_sentiment'] = product_reviews['text_sentiment'].fillna('neutral')

    # Filter by sentiment
    if sentiment in ('positive','neutral', 'negative'):
        sentiment_reviews = product_reviews[product_reviews['text_sentiment'] == sentiment]
    else:
        raise ValueError(f"Invalid sentiment: {sentiment}. Must be 'positive', 'neutral', or 'negative'")

    # Handle edge case: no reviews in this sentiment category
    if len(sentiment_reviews) == 0:
        return []

    # Count word frequencies across all reviews
    word_counter = Counter()

    for text in sentiment_reviews['processed_review_text']:
        # Skip NaN or empty values
        if pd.isna(text) or text == '':
            continue

        # Split space-separated tokens and count
        tokens = text.split()
        word_counter.update(tokens)

    # Return top N terms
    return word_counter.most_common(top_n)

### Comparison Chart: Function `create_multi_comparison_chart` (`sentiment.py`)
This function uses `plotly` library to group multiple bar charts. A limit of 1-5 products, color scheme, and dynamic legend sizing are set to ensure readability.

The **5-product cap** comes from 3 constraints: 
- The color palette defines 5 distinct colors — beyond this products share colors and become indistinguishable
- The 2-column subplot grid fits 5 products in 3 rows without excessive scrolling
- Human comparison ability degrades sharply beyond 5-7 items per chart.

In [ ]:
COMPARISON_COLORS = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']


def create_multi_comparison_chart(
    breakdowns: List[Dict],
    names: List[str]
) -> go.Figure:
    """
    Create a grouped bar chart comparing sentiment distributions of 2-5 products.
    """
    categories = ['Positive', 'Neutral', 'Negative']

    fig = go.Figure()

    for i, (breakdown, name) in enumerate(zip(breakdowns, names)):
        pcts = [breakdown['positive_pct'], breakdown['neutral_pct'], breakdown['negative_pct']]
        fig.add_trace(go.Bar(
            x=categories,
            y=pcts,
            name=f'{name} (n={breakdown["total"]} reviews)',
            marker_color=COMPARISON_COLORS[i % len(COMPARISON_COLORS)],
            text=[f'{p:.1f}%' for p in pcts],
            textposition='outside',
            textfont=dict(color='black', size=11)
        ))

    n = len(breakdowns)
    legend_rows = math.ceil(n / 2)
    extra_legend_height = max(0, legend_rows - 1) * 28
    chart_height = 420 + extra_legend_height
    bottom_margin = 80 + extra_legend_height

    fig.update_layout(
        title=dict(
            text='Sentiment Comparison',
            font=dict(size=16, family='-apple-system, BlinkMacSystemFont, sans-serif'),
            x=0.5
        ),
        barmode='group',
        yaxis=dict(title='Percentage of Reviews', range=[0, 100], gridcolor='#f0f0f0'),
        xaxis=dict(title=''),
        legend=dict(orientation='h', yanchor='top', y=-0.12, xanchor='center', x=0.5),
        margin=dict(t=60, b=bottom_margin, l=50, r=20),
        height=chart_height,
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor='rgba(0,0,0,0)',
        font=dict(family='-apple-system, BlinkMacSystemFont, sans-serif'),
        bargap=0.2,
        bargroupgap=0.1
    )

    return fig

### User Interface Integration: Function `build_sentiment_tab()` + `on_analyze()` (`sentiment_tab.py`)
This function uses `gradio` to help build the user interface.
- Multi-select dropdown helps the user to select and compare multiple products
- Three outputs:
    - Sentiment Comparison: showing the sentiment distribution of all selected products
    - Individual Analysis: showing the sentiment distribution of one product at a time
    - Key Terms Table: showing key terms in positive and negative reviews in one product at a time

**An important property of this implementation**: the UI tab required zero changes when switching from **rating-based (approach A)** to **text-based (approach B)** sentiment. Because `get_sentiment_breakdown()` and `get_key_terms()` keep identical signatures and return identical structures, the entire change is encapsulated in the backend.

In [ ]:
def build_sentiment_tab(product_choices, df, products_df):
    """Build the Sentiment Insights tab UI and wire event handlers."""

    with gr.Tab("Sentiment Insights", elem_id="tab-sentiment", elem_classes=["glamreview-tab"]):
        gr.Markdown("### Analyze sentiment and key terms from product reviews")
        gr.Markdown("Select one or more products to analyze and compare their sentiment.")

        product_selector = gr.Dropdown(
            choices=product_choices,
            label="Select Products",
            multiselect=True,
            filterable=True,
            value=[],
        )

        analyze_btn = gr.Button("Analyze", variant="primary")

        # Outputs
        comparison_chart = gr.Plot(label="Sentiment Comparison")
        sentiment_chart = gr.Plot(label="Individual Analysis")
        key_terms_html = gr.HTML()

        def on_analyze(selected_products):
            if not selected_products:
                return [None, None, ""]

            # Deduplicate (shouldn't happen with multiselect, but be safe)
            seen = set()
            active_ids = []
            for pid in selected_products:
                if pid and pid not in seen:
                    seen.add(pid)
                    active_ids.append(pid)

            if not active_ids:
                return [None, None, ""]

            new_reviews_df = pd.DataFrame(new_reviews) if new_reviews else None

            def get_name(pid):
                row = products_df[products_df['product_id'] == pid].iloc[0]
                return f"{row['brand_name']} - {row['product_title']}"

            breakdowns = []
            names = []
            for pid in active_ids:
                bd = get_sentiment_breakdown(pid, df, new_reviews_df)
                breakdowns.append(bd)
                names.append(get_name(pid))

            # Comparison chart (2+ products)
            comp_fig = create_multi_comparison_chart(breakdowns, names) if len(active_ids) >= 2 else None

            # Individual bar charts in 2-col grid
            n = len(active_ids)
            cols = min(n, 2)
            rows = math.ceil(n / cols)
            short_titles = [
                (name[:25] + "…") if len(name) > 25 else name
                for name in names
            ]

            fig = make_subplots(
                rows=rows, cols=cols,
                subplot_titles=[f"{short_titles[i]} (n={breakdowns[i]['total']} reviews)" for i in range(n)],
                vertical_spacing=0.22,
                horizontal_spacing=0.12,
            )

            categories = ['Positive', 'Neutral', 'Negative']
            bar_colors = ['#2ecc71', '#95a5a6', '#EE4D2D']

            for i, bd in enumerate(breakdowns):
                r = i // cols + 1
                c = i % cols + 1
                pcts = [bd['positive_pct'], bd['neutral_pct'], bd['negative_pct']]
                fig.add_trace(
                    go.Bar(
                        x=categories, y=pcts,
                        marker_color=bar_colors,
                        text=[f'{p:.1f}%' for p in pcts],
                        textposition='inside',
                        textfont=dict(color='white', size=11),
                        showlegend=False,
                    ),
                    row=r, col=c,
                )
                fig.update_yaxes(range=[0, 100], gridcolor='#f0f0f0', row=r, col=c)

            fig.update_layout(
                height=300 * rows,
                margin=dict(t=50, b=30, l=40, r=20),
                paper_bgcolor='rgba(0,0,0,0)',
                plot_bgcolor='rgba(0,0,0,0)',
                font=dict(family='-apple-system, BlinkMacSystemFont, sans-serif'),
                showlegend=False,
            )

            # Key terms for all products
            terms_sections = []
            for i, pid in enumerate(active_ids):
                pos_terms = get_key_terms(pid, df, 'positive', top_n=15, new_reviews=new_reviews_df)
                neg_terms = get_key_terms(pid, df, 'negative', top_n=15, new_reviews=new_reviews_df)
                pos_html = render_key_terms_table(pos_terms, "positive")
                neg_html = render_key_terms_table(neg_terms, "negative")

                section = f"""
                <div style="margin-bottom: 24px; padding: 16px; border: 1px solid #e0e0e0; border-radius: 8px;">
                    <h4 style="margin: 0 0 12px 0; color: #333;">{names[i]}</h4>
                    <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 16px;">
                        <div>
                            <h5 style="margin: 0 0 8px 0;">😊 Positive Terms</h5>
                            {pos_html}
                        </div>
                        <div>
                            <h5 style="margin: 0 0 8px 0;">😞 Negative Terms</h5>
                            {neg_html}
                        </div>
                    </div>
                </div>
                """
                terms_sections.append(section)

            return [comp_fig, fig, "".join(terms_sections)]

        analyze_btn.click(
            fn=on_analyze,
            inputs=[product_selector],
            outputs=[comparison_chart, sentiment_chart, key_terms_html],
            show_progress="hidden",
        )

### Results & Analysis

#### Distribution
When the app starts, `compute_text_sentiment_cached()` prints the overall sentiment distribution across all ~60,000 reviews. For our dataset, the distribution is roughly:
| Class | Count | Percent |
|---|---|---|
| positive | 42,611 | 69.5% |
| neutral | 7,422 | 12.1% |
| negative | 11,242 | 18.3% |

Three observations are worth noting:

- The **positive** proportion is lower than Milestone 1's buyer prevalence **(78.7%)**. This is expected: a probability ≥ 0.60 is stricter than a 50% `predict()` call, so some reviews the model would have labeled "buyer" land in the neutral zone.
- The **neutral** fraction reflects the model's uncertainty, not actual neutral sentiment. This is an interpretive limitation worth flagging.
- The **negative** fraction is smaller than the 21.3% non-buyer rate from Milestone 1, again because the 0.40 threshold is stricter than 0.50.

#### Individual Analysis
As a worked example, consider the **L'Oreal Paris Infallible Le Rouge Lipstick (n=319 reviews)**:

- **Sentiment breakdown**: positive 20.4%, neutral 16.0%, negative 63.6%
- **Top positive terms**: lipstick (12), tone (9), price (6), stay (6), dark (4)
- **Top negative terms**: lipstick (131), stay (49), texture (48), lasting (33), matte (31)

The high negative proportion is striking. A **rating-based** view would likely show this product as mostly positive (consistent with the corpus-wide rating skew). The **text-based** view, by contrast, picks up linguistic patterns in the reviews — words like **lipstick** dominating the negative key terms, which align with what RF + unweighted GloVe + metadata model pipeline learned during Milestone 1 training.

#### Sentiment Comparison (Multiple Products)
The text-based approach is at its most useful when comparing multiple products side by side. Selecting five products from different brands and categories produced the following distribution:

| Product | Positive | Neutral | Negative | n |
|---|---|---|---|---|
| Nivea Sun Lotion SPF 50 | 88.7% | 6.5% | 4.8% | 771 |
| Maybelline Face Studio Master Contour Palette | 31.1% | 23.1% | 45.8% | 489 |
| L'Oreal Infallible Le Rouge Lipstick | 20.4% | 16.0% | 63.6% | 319 |
| Lakme Absolute White Intense SPF 20 Concealer Stick | 18.8% | 19.4% | 61.8% | 458 |
| NYX Professional Makeup Pore Filler | 12.0% | 20.7% | 67.3% | 150 |

The variance is noticeable. Nivea's sunscreen sits at **88.7%** positive, the NYX makeup pore filler at **12.0%**. This spread of **76 percentage points** would be heavily flattened in a rating-based view, where the corpus skew makes nearly every product appear "positive." The text-based view separates products by what reviewers write, not whether they click 4 or 5 stars.

### Limitations

**Session-only new reviews**: Reviews submitted during a running session are classified as 'neutral' until the next app restart. The cleaner fix is to add a single model C inference call inside the review-submission handler in `classifier.py`.

**Thresholds were chosen by heuristic, not tuned**: The 0.60/0.40 cutoffs were selected to be symmetric around 0.50 and leave a 0.20-wide neutral zone, but were not validated against labeled sentiment data (the dataset has buyer/non-buyer labels, not sentiment labels).

**Unigram-only key terms**: Frequency counting treats each word independently. Bigrams ("not great", "highly recommend") carry sentiment information that unigrams miss. The collocation dictionary already in the repo (`collocation_dict.pkl`) could be leveraged to extract phrases.

### Conclusion

Task 4 implements a text-based sentiment analysis dashboard using Milestone 1's best-performing model **(Random Forest + Unweighted GloVe + Title + Metadata, macro F1 = 0.6808)** to analyze sentiment on one or multiple products. The approach surfaces meaningful between-product differences, a 76 percentage-point spread between the most positively and most negatively received products in our sample, providing brands with **actionable insights** into customer reception that rating-based analysis would have flattened.